# SMAP / HydroBlocks metadata and time-reference check

1. Prints raster metadata for the SMAP NSIDC-0779 GeoTIFF and the HydroBlocks `hrus.vrt` domain raster.
2. Checks whether HydroBlocks' postprocessed `t` time axis is **UTC** or **local solar time (LST)**, using `cosz` sunrise/sunset timing.

Sunrise/sunset in local solar time depends only on latitude and day-of-year, never on longitude. So:
- If `t` is true UTC, converting to LST via `+lon/15` should make sunrise/sunset line up at nearly the same LST hour across every longitude in the domain.
- If `t` is already local solar time, applying `+lon/15` on top of that would double-correct, and sunrise/sunset would appear to *drift* across longitude instead of lining up.

Whichever hypothesis gives a small, stable spread close to the independently-computed solar-geometry expectation is correct.

## Imports

In [1]:
import math
import os
from datetime import datetime

import netCDF4 as nc
import numpy as np
import rasterio

## 1. SMAP soil moisture file metadata

In [2]:
smap_path = (
    "/scratch/alpine/battobrah@xsede.org/HB_NMP_full_domain_comparison_data/"
    "SMAP_0779_2015_2024_full domain/"
    "NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20150401.tif"
)

with rasterio.open(smap_path) as src:
    print("=== SMAP NSIDC-0779 ===")
    print(f"CRS: {src.crs}")
    print(f"Transform: {src.transform}")
    print(f"Resolution (x, y): {src.res}")
    print(f"Shape (height, width): {src.shape}")
    print(f"Bounds: {src.bounds}")
    print(f"Dtype: {src.dtypes}")
    print(f"Nodata: {src.nodata}")

RasterioIOError: /scratch/alpine/battobrah@xsede.org/HB_NMP_full_domain_comparison_data/SMAP_0779_2015_2024_full domain/NSIDC-0779_EASE2_G1km_SMAP_SM_DS_20150401.tif: No such file or directory

## 2. HydroBlocks domain raster (hrus.vrt) metadata

In [ ]:
hb_path = (
    "/scratch/alpine/battobrah@xsede.org/Rice_NMT_Snow/upper_colorado/"
    "experiments/simulations/"
    "MSWX_3h_Snow_Project3_3d_upper_colorado_11yr_50hru_full_domain_downscaled_test/"
    "postprocess/hrus.vrt"
)

with rasterio.open(hb_path) as src:
    print("=== HydroBlocks (hrus.vrt) ===")
    print(f"CRS: {src.crs}")
    print(f"Transform: {src.transform}")
    print(f"Resolution (x, y): {src.res}")
    print(f"Shape (height, width): {src.shape}")
    print(f"Bounds: {src.bounds}")
    print(f"Dtype: {src.dtypes}")
    print(f"Nodata: {src.nodata}")

## 3. Check: is HydroBlocks' postprocessed time axis UTC or LST?

### Configuration
Change `CHECK_DATE` to try other dates -- a solstice date (e.g. `"20150621"`) makes the UTC-vs-LST distinction most obvious, since day length is furthest from 12 hours then.

In [ ]:
CHECK_DATE = "20150401"   # match the SMAP file's date above
N_COLUMNS = 6              # how many longitude columns to sample
ROW_INDEX = None           # None -> use the middle row of the domain

EDIR = (
    "/scratch/alpine/battobrah@xsede.org/Rice_NMT_Snow/upper_colorado/"
    "experiments/simulations/"
    "MSWX_3h_Snow_Project3_3d_upper_colorado_11yr_50hru_full_domain_downscaled_test"
)

### Solar-geometry helper functions

In [ ]:
def julian_day(date):
    return date.timetuple().tm_yday


def solar_declination_deg(julian):
    """Standard approximation (Cooper 1969), degrees."""
    return 23.45 * math.sin(math.radians(360.0 / 365.0 * (284 + julian)))


def expected_sunrise_sunset_lst(latitude_deg, julian):
    """Expected sunrise/sunset in local solar time (hours), from the
    standard hour-angle formula."""
    declination = math.radians(solar_declination_deg(julian))
    lat = math.radians(latitude_deg)
    cos_hour_angle = -math.tan(lat) * math.tan(declination)
    cos_hour_angle = max(-1.0, min(1.0, cos_hour_angle))
    hour_angle_deg = math.degrees(math.acos(cos_hour_angle))
    half_day_length_hours = hour_angle_deg / 15.0
    return 12.0 - half_day_length_hours, 12.0 + half_day_length_hours


def find_zero_crossings(hours, cosz):
    """Return (rising_hour, falling_hour) where cosz crosses zero, via
    linear interpolation between the two bracketing samples."""
    rising = None
    falling = None
    for i in range(len(cosz) - 1):
        if cosz[i] <= 0.0 < cosz[i + 1]:
            weight = -cosz[i] / (cosz[i + 1] - cosz[i])
            rising = hours[i] + weight * (hours[i + 1] - hours[i])
        elif cosz[i] > 0.0 >= cosz[i + 1]:
            weight = cosz[i] / (cosz[i] - cosz[i + 1])
            falling = hours[i] + weight * (hours[i + 1] - hours[i])
    return rising, falling

### Read the HydroBlocks file and inspect the 't' variable / global attributes

In [ ]:
hb_folder = os.path.join(EDIR, "postprocess", "output_dir")
hb_nc_path = os.path.join(hb_folder, f"{CHECK_DATE}.nc")

if not os.path.exists(hb_nc_path):
    raise FileNotFoundError(f"No postprocessed file found for {CHECK_DATE}: {hb_nc_path}")

with nc.Dataset(hb_nc_path) as dataset:
    lon = np.asarray(dataset.variables["lon"][:], dtype=float)
    lat = np.asarray(dataset.variables["lat"][:], dtype=float)
    cosz = np.ma.asarray(dataset.variables["cosz"][:]).filled(np.nan)

    print("--- 't' variable metadata (if present) ---")
    if "t" in dataset.variables:
        t_var = dataset.variables["t"]
        print("Raw values:", np.asarray(t_var[:]))
        for attribute_name in t_var.ncattrs():
            print(f"  {attribute_name} = {getattr(t_var, attribute_name)}")
    else:
        print("No 't' variable found in this file.")

    print("\n--- Global attributes (looking for timezone/UTC hints) ---")
    for attribute_name in dataset.ncattrs():
        print(f"  {attribute_name} = {getattr(dataset, attribute_name)}")

### Sort coordinates and set up the per-column comparison

In [ ]:
lon_order = np.argsort(lon)
lat_order = np.argsort(lat)
lon = lon[lon_order]
lat = lat[lat_order]
cosz = cosz[:, lat_order, :][:, :, lon_order]

n_time = cosz.shape[0]
assumed_utc_hours = np.array([3 * i for i in range(n_time)], dtype=float)
print(f"Assumed stored UTC hours for the {n_time} daily slots: {assumed_utc_hours}")

row_index = ROW_INDEX if ROW_INDEX is not None else len(lat) // 2
latitude = lat[row_index]
date = datetime.strptime(CHECK_DATE, "%Y%m%d")
julian = julian_day(date)
expected_rise, expected_set = expected_sunrise_sunset_lst(latitude, julian)

print(f"\nDate: {date.date()}  (day of year {julian})")
print(f"Sample row latitude: {latitude:.4f} deg")
print(
    f"Expected sunrise/sunset in LOCAL SOLAR TIME at this latitude "
    f"(from solar geometry): {expected_rise:.2f}h / {expected_set:.2f}h"
)

### Run the check across several longitude columns

In [ ]:
column_indices = np.linspace(0, len(lon) - 1, N_COLUMNS).astype(int)

header = (
    f"{'lon':>9} | {'sunrise if t=UTC->LST':>22} | "
    f"{'sunset if t=UTC->LST':>21} | {'sunrise if t=LST already':>25} | "
    f"{'sunset if t=LST already':>24}"
)
print(header)
print("-" * len(header))

utc_rises, utc_sets = [], []
raw_rises, raw_sets = [], []

for column in column_indices:
    longitude = lon[column]
    series = cosz[:, row_index, column]
    if not np.any(np.isfinite(series)):
        continue

    # Hypothesis A: stored hours are true UTC; convert to LST.
    lst_hours_hyp_a = assumed_utc_hours + longitude / 15.0
    rise_a, set_a = find_zero_crossings(lst_hours_hyp_a, series)

    # Hypothesis B: stored hours are already local solar time.
    rise_b, set_b = find_zero_crossings(assumed_utc_hours, series)

    if rise_a is not None:
        utc_rises.append(rise_a)
    if set_a is not None:
        utc_sets.append(set_a)
    if rise_b is not None:
        raw_rises.append(rise_b)
    if set_b is not None:
        raw_sets.append(set_b)

    def fmt(value):
        return f"{value:.2f}h" if value is not None else "n/a"

    print(
        f"{longitude:9.3f} | {fmt(rise_a):>22} | {fmt(set_a):>21} | "
        f"{fmt(rise_b):>25} | {fmt(set_b):>24}"
    )

### Verdict

In [ ]:
def spread(values):
    return (max(values) - min(values)) if len(values) >= 2 else float("nan")


print("=== Verdict ===")
print(
    f"Hypothesis A (t = UTC, converted via +lon/15): "
    f"sunrise spread across longitude = {spread(utc_rises):.2f}h, "
    f"sunset spread = {spread(utc_sets):.2f}h"
)
print(
    f"Hypothesis B (t already = local solar time):   "
    f"sunrise spread across longitude = {spread(raw_rises):.2f}h, "
    f"sunset spread = {spread(raw_sets):.2f}h"
)
print(
    "\nA physically correct interpretation should show a SMALL spread "
    "(sunrise/sunset shouldn't depend on longitude at fixed latitude/"
    "date) and should be close to the solar-geometry expectation "
    f"printed above ({expected_rise:.2f}h / {expected_set:.2f}h)."
)
print(
    "Whichever hypothesis has the smaller spread AND is closer to the "
    "solar-geometry expectation is the correct interpretation of the "
    "stored 't' axis."
)